# 09. 이상 라벨 20%인 학습 데이터 만들기

`cal01_base_v1.csv`를 학습 80%, 평가 20%로 나눈 뒤 학습 데이터만 0:1 = 4:1로 맞춤.
라벨 1은 임시 이상 후보, 0은 기존 규칙에 걸리지 않은 행임. 확정된 정상, 고장 정답은 아님.

In [ ]:
import os
import pandas as pd

ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)

DATA_DIR = os.path.join(ROOT, "data")
if os.path.isfile(os.path.join(DATA_DIR, "raw", "T-CR1-CAL01_온도.csv")):
    DATA_DIR = os.path.join(DATA_DIR, "raw")

PROCESSED_DIR = os.path.join(ROOT, "data", "processed")
FIG_DIR = os.path.join(ROOT, "figures")
ENC = "utf-8-sig"

TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")
WELD_CSV = os.path.join(DATA_DIR, "T-CR1-WLD01_용접.csv")
BASE_CSV = os.path.join(PROCESSED_DIR, "cal01_base_v1.csv")

from sklearn.model_selection import train_test_split
from IPython.display import display

TEST_SIZE = 0.20
ANOMALY_RATIO = 0.20
RANDOM_STATE = 42

OUT_DIR = os.path.join(PROCESSED_DIR, "balanced_20pct")
TRAIN_CSV = os.path.join(OUT_DIR, "train.csv")
TEST_CSV = os.path.join(OUT_DIR, "test.csv")
SPLIT_CSV = os.path.join(OUT_DIR, "row_split.csv")
COUNTS_CSV = os.path.join(OUT_DIR, "label_counts.csv")


## 1. 기존 라벨이 있는 행 읽기

사용자 제외 행과 라벨이 없는 행을 뺌. 기존 값, 결측, 라벨은 유지함.
`MODEL_ELIGIBLE`은 기존 67개 입력의 완성 여부라 여기서 추가 필터로 쓰지 않음. 실제 입력 컬럼은 모델에서 정함.

In [ ]:
# 저장 후에도 원본 소수값을 유지하도록 읽음.
base = pd.read_csv(BASE_CSV, encoding=ENC, float_precision="round_trip", low_memory=False)
base["MEAS_DT"] = pd.to_datetime(base["MEAS_DT"], errors="raise")

data = base.loc[
    base["EXCLUDED"].eq(False) & base["LABEL"].isin([0, 1])
].copy()

data = data.sort_values(["LOT_NO", "MEAS_DT"]).reset_index(drop=True)
y = data["LABEL"]


## 2. 학습과 평가를 먼저 나누기

`stratify=y`로 라벨 0, 1의 비율을 양쪽에 비슷하게 유지함. 강종별 비율을 맞추는 설정은 아님.
`random_state=42`는 다시 실행해도 같은 행을 뽑기 위한 값임. 평가 데이터는 이후 줄이거나 늘리지 않음.


In [ ]:
train_original, test = train_test_split(
    data,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

train_original = train_original.sort_values(["LOT_NO", "MEAS_DT"]).reset_index(drop=True)
test = test.sort_values(["LOT_NO", "MEAS_DT"]).reset_index(drop=True)


## 3. 강종 비중을 유지하면서 이상 비율을 20%로 맞추기

학습의 라벨 1은 모두 남김. 라벨 0은 라벨 1의 4배만 뽑되, 기존 학습의 강종별 라벨 0 비중을 유지함.
강종별 개수는 정수로 맞추므로 비례 계산값과 1행 미만 차이가 남. 각 강종의 이상 비율을 20%로 맞추는 것은 아님.
뽑은 행은 LOT별 측정 시각순으로 정렬함. 중간 행이 빠지므로 연속된 시계열은 아니며, 과거 파생값은 원본 값을 유지함.


In [ ]:
label_0 = train_original.loc[train_original["LABEL"].eq(0)]
label_1 = train_original.loc[train_original["LABEL"].eq(1)]

n_label_0 = round(len(label_1) * (1 - ANOMALY_RATIO) / ANOMALY_RATIO)

# 학습의 라벨 0 안에서 강종 비중에 맞춰 남길 행을 뽑음.
sampled_0, _ = train_test_split(
    label_0,
    train_size=n_label_0,
    random_state=RANDOM_STATE,
    stratify=label_0["GRADE"],
)

train_balanced = pd.concat([sampled_0, label_1])
train_balanced = train_balanced.sort_values(["LOT_NO", "MEAS_DT"]).reset_index(drop=True)


## 4. 전후 개수와 강종, LOT별 구성 확인

학습의 라벨 1 비율이 20%인지 확인함. 강종별 표는 라벨 0을 뽑기 전후의 비중과 남긴 라벨 1 개수를 비교함.
강종별 비중이 비슷해도 드문 센서 변화까지 남았다는 뜻은 아님. LOT별 표도 함께 확인함.


In [ ]:
datasets = {
    "라벨이 있는 원본": data,
    "학습, 비율 조정 전": train_original,
    "학습, 이상 20%": train_balanced,
    "평가, 원래 비율": test,
}

rows = []

for name, frame in datasets.items():
    rows.append({
        "데이터": name,
        "전체 행": len(frame),
        "라벨 0": int(frame["LABEL"].eq(0).sum()),
        "라벨 1": int(frame["LABEL"].eq(1).sum()),
        "라벨 1 비율(%)": round(frame["LABEL"].eq(1).mean() * 100, 4),
    })

label_counts = pd.DataFrame(rows)
display(label_counts)

grade_counts = pd.DataFrame({
    "뽑기 전 라벨 0": label_0.groupby("GRADE").size(),
    "뽑은 라벨 0": sampled_0.groupby("GRADE").size(),
    "남긴 라벨 1": label_1.groupby("GRADE").size(),
}).fillna(0).astype(int)

grade_counts["뽑기 전 라벨 0 안에서 비중(%)"] = (grade_counts["뽑기 전 라벨 0"] / len(label_0) * 100).round(2)
grade_counts["뽑은 라벨 0 안에서 비중(%)"] = (grade_counts["뽑은 라벨 0"] / len(sampled_0) * 100).round(2)
grade_counts = grade_counts.reset_index().rename(columns={"GRADE": "강종"})
display(grade_counts)

lot_tables = []

for name, frame in datasets.items():
    counts = pd.crosstab(frame["LOT_NO"], frame["LABEL"])
    counts = counts.reindex(columns=[0, 1], fill_value=0)
    counts.columns = ["라벨 0", "라벨 1"]
    counts = counts.reset_index()
    counts.insert(0, "데이터", name)
    lot_tables.append(counts)

lot_counts = pd.concat(lot_tables, ignore_index=True)
display(lot_counts)


## 5. CSV 저장

`data/processed/balanced_20pct/`에 저장함.

| 파일 | 내용 |
|---|---|
| `train.csv` | 라벨 1이 20%인 학습 데이터 |
| `test.csv` | 원래 라벨 비율을 유지한 평가 데이터 |
| `row_split.csv` | 라벨이 있는 행의 학습, 평가 구분과 학습에 남긴 여부 |
| `label_counts.csv` | 비율 조정 전후 개수 |

`train.csv`, `test.csv`는 원본의 106개 컬럼을 유지함. 전체 컬럼을 그대로 모델 입력 X에 넣으면 안 됨. `LABEL`은 y이고, 라벨 근거와 사후 정비 정보는 조회용임.

In [ ]:
row_split = data[["ROW_ID", "LOT_NO", "MEAS_DT", "LABEL"]].copy()
row_split["SPLIT"] = "train"
row_split.loc[row_split["ROW_ID"].isin(test["ROW_ID"]), "SPLIT"] = "test"
row_split["IN_BALANCED_TRAIN"] = row_split["ROW_ID"].isin(train_balanced["ROW_ID"])

os.makedirs(OUT_DIR, exist_ok=True)

train_balanced.to_csv(TRAIN_CSV, index=False, encoding=ENC)
test.to_csv(TEST_CSV, index=False, encoding=ENC)
row_split.to_csv(SPLIT_CSV, index=False, encoding=ENC)
label_counts.to_csv(COUNTS_CSV, index=False, encoding=ENC)

print("저장 폴더:", os.path.relpath(OUT_DIR, ROOT))
print("학습:", train_balanced.shape, "/ 평가:", test.shape)


행을 무작위로 나누므로 같은 LOT의 가까운 시각이 학습과 평가에 함께 들어갈 수 있음. 시간순으로 다시 정렬해도 이 점은 같음.
이번 파일은 현재 자료 안에서 임시 라벨을 분류하는 비교용임. 미래 기간의 고장 예측 성능은 별도 시간 분할로 확인해야 함.
빠진 행이 있으므로 이 파일에서 이동평균이나 직전 값 같은 과거 파생값을 다시 만들지 않음.

참고: [stratify 설명](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html), [학습 데이터에서만 비율 조정](https://imbalanced-learn.org/stable/common_pitfalls.html).
